# Stage-4 v2 — training-input cache (research record §44)

Builds, once, everything Stage-4 v2 training reads. **No training, no Stage-4 prediction cache, no IDRiD test images.**

1. Setup (Drive, repo, `TJDR_RAW_DIR`/`TJDR_PROCESSED_DIR`). No GPU needed.
2. TJDR Stage-2 DR preprocessing, **on demand** (only for images whose cache sample is still missing) → `datasets/TJDR/processed/` on Drive (canonical `image_preprocessing.preprocess_image`, lossless PNG) + exact parity check.
3. IDRiD: reuse the existing Drive `datasets/IDRiD/segmentation/processed/` (Stage-2 output) + parity check.
4. Build `cache/Stage4Train/s4train-v1/` on Drive: Stage-2 image → 1536² uint8 + 4 binary masks at 1536 (IDRiD 44 train / 10 val; TJDR 443 train / 110 val). **Resumable: files already on Drive whose SHA matches `progress.json` are skipped; a missing or truncated file is rebuilt.**
5. Verify the cache (counts, split sha, per-file sha, fingerprint).

In [ ]:
# ==== [1] SETUP (Drive, repo, requirements incl. smp 0.5.0, TJDR/IDRiD paths) ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import stage4_v2_setup
SETUP_INFO = stage4_v2_setup.setup_stage4_v2(fetch_weights=False)

In [ ]:
# ==== [2] TJDR STAGE-2 ON DEMAND -> datasets/TJDR/processed (resumable; only images still needed) ====
import cv2
import numpy as np
import image_preprocessing as ip
import stage4_v2_data as data
import tjdr_dataset as tj


def tjdr_processed_on_demand(split, image_id):
    """Stage-2 DR output of one TJDR image: created once (lossless PNG) when missing, then reused."""
    tj.assert_usable(split, image_id)
    out = tj.processed_image_path(split, image_id)
    if not os.path.exists(out):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        ip.preprocess_image(tj.image_path(split, image_id), out + ".tmp.png", profile="DR")
        os.replace(out + ".tmp.png", out)
    return out

In [ ]:
# ==== [3] IDRiD: existing Stage-2 outputs (processed/, JPEG) -- presence + parity ====
split = data.idrid_split()
proc = os.path.join(data.idrid_processed_dir(), "1. Original Images", "a. Training Set")
missing = [i for i in split["train"] + split["val"] if not os.path.exists(os.path.join(proc, f"{i}.jpg"))]
assert not missing, f"IDRiD processed images missing: {missing}"
raw = os.path.join(data.idrid_raw_dir(), "1. Original Images", "a. Training Set")
for image_id in ("IDRiD_01", "IDRiD_20", "IDRiD_40"):
    par = data.stage2_parity(os.path.join(raw, f"{image_id}.jpg"), os.path.join(proc, f"{image_id}.jpg"))
    print(image_id, par)
    assert par["psnr"] >= 30, "IDRiD processed image does not match Stage 2 (beyond JPEG loss)"

In [ ]:
# ==== [4] BUILD cache/Stage4Train/s4train-v1 ON DRIVE (resumable, ~30-60 min CPU) ====
CACHE_DIR = data.cache_dir()
print("cache dir:", CACHE_DIR)


def loader(entry):
    dataset, role, split, image_id, _ = entry
    if dataset == "IDRiD":
        return data.load_idrid_pair(image_id)
    tjdr_processed_on_demand(split, image_id)
    return tj.load_pair(split, image_id, stage2=True)


ENTRIES = data.source_entries()
print(len(ENTRIES), "entries")
MANIFEST = data.build_training_cache(CACHE_DIR, ENTRIES, loader)
print("counts:", MANIFEST["counts"], "complete:", MANIFEST["complete"], "fingerprint:", MANIFEST["fingerprint"])

In [ ]:
# ==== [5] VERIFY (incl. bit-exact Stage-2 parity on TJDR images created in this session) ====
for _split, _id in (("train", "TJDR_train_001"), ("train", "TJDR_train_300"), ("test", "TJDR_test_050")):
    if os.path.exists(tj.processed_image_path(_split, _id)):
        _par = data.stage2_parity(tj.image_path(_split, _id), tj.processed_image_path(_split, _id))
        print(_id, _par)
        assert _par["max_abs"] == 0.0, "TJDR Stage-2 output is not bit-exact"
tc = data.TrainingCache(CACHE_DIR, verify_files=True)
print({f"{d}/{r}": len(tc.names(r, d)) for d in ("IDRiD", "TJDR") for r in ("train", "val")})
print("fingerprint:", tc.fingerprint)